# Notebook 15: Minimal End-to-End Pipeline for User Data

**Goal:** A practical guide for a new user wanting to apply PhosCrosstalk to their own data.

Every step is shown with the sample data as a concrete runnable example.

## 1. Data Preparation Checklist

| File | Required | Description |
|------|----------|-------------|
| `protephospho.csv` | ✅ | GeneID, Psite, x1..xN |
| `kinase_sites.tsv` | ✅ | Site (`Gene_Psite`), Kinase, weight |
| `mrna.csv` | ❌ optional | GeneID, x1..xN mRNA time series |
| `tf_mrna.csv` | ❌ optional | Source, Target, Weight TF network |

**Column naming rules:**
- Time columns: `x1, x2, …, xN` in order
- `GeneID`/`Protein`: protein identifier
- `Psite`: site label e.g. `EGFR_Y1068`
- Kinase names must match `GeneID` values in `protephospho.csv`

## 2. Minimal `config.toml`

```toml
[paths]
data        = "data/protephospho.csv"
kinase_tsv  = "data/kinase_sites.tsv"
output_dir  = "results/"
# Optional:
# mrna_csv  = "data/mrna.csv"
# tf_net    = "data/tf_mrna.csv"

[time]
phosphosite_time_points = [0,1,2,5,10,15,20,30,45,60,90,120,180,240]
mrna_time_points        = [0,1,2,5,10,15,20,30,45,60,90,120,180,240]
protein_time_points     = [0,1,2,5,10,15,20,30,45,60,90,120,180,240]
```

The number of values **must match** the number of `x1..xN` columns in your CSVs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print("Setup complete.")

## 3a. Inspect Raw Files

In [ ]:
TIMEPOINTS = list(range(1, 15))   # 14 time points: x1..x14

phospho_df = pd.read_csv(SAMPLE_DIR / 'protephospho.csv')
kinase_df  = pd.read_csv(SAMPLE_DIR / 'kinase_sites.tsv', sep='\t')

# Optional files — guard with exists() so the workflow runs for users without them
mrna_path = SAMPLE_DIR / 'mrna.csv'
tf_path   = SAMPLE_DIR / 'tf_mrna.csv'
mrna_df = pd.read_csv(mrna_path) if mrna_path.exists() else None
tf_df   = pd.read_csv(tf_path)   if tf_path.exists()   else None

print('protephospho.csv', phospho_df.shape)
print(phospho_df.head(3).to_string())
print()
print('kinase_sites.tsv', kinase_df.shape)
print(kinase_df.head(3).to_string())
print()
print('mrna.csv     :', mrna_df.shape if mrna_df is not None else 'not found (optional)')
print('tf_mrna.csv  :', tf_df.shape   if tf_df  is not None else 'not found (optional)')


## 3b. Data Quality Checks

In [ ]:
x_cols = [c for c in phospho_df.columns if c.startswith('x')]
n_tp_phospho = len(x_cols)

nan_count = phospho_df[x_cols].isna().sum().sum()
inf_count = np.isinf(phospho_df[x_cols].values.astype(float)).sum()
print(f'NaN in phospho data : {nan_count}')
print(f'Inf in phospho data : {inf_count}')
print(f'Phospho time points : {n_tp_phospho}')

if mrna_df is not None:
    n_tp_mrna = len([c for c in mrna_df.columns if c.startswith('x')])
    print(f'mRNA   time points  : {n_tp_mrna}')
    assert n_tp_phospho == n_tp_mrna, 'Time point count mismatch!'
    print('\u2713 Phospho / mRNA time point counts match')
else:
    print('mRNA data not present (optional) \u2014 skipping mRNA time-point check')


In [ ]:
all_genes   = set(phospho_df['GeneID'].unique())
tsv_kinases = set(kinase_df['Kinase'].unique())
overlap = all_genes & tsv_kinases
print(f'Genes in phospho    : {sorted(all_genes)}')
print(f'Kinases in tsv      : {sorted(tsv_kinases)}')
print(f'Overlap             : {sorted(overlap)}')
if not overlap:
    print('WARNING: no overlap — K_site_kin will be all-zero!')
else:
    print('✓ Kinase overlap is non-empty')

## 3c. Load Data with the API

In [ ]:
from phoscrosstalk.data_loader import (
    load_site_data, load_rna_data, load_kinase_site_matrix,
    load_tf_network, build_tf_prot_weights, apply_scaling, row_normalize,
)

sites, proteins, site_prot_idx, positions, t_phos, Y, A_data, A_proteins = \
    load_site_data(str(SAMPLE_DIR / 'protephospho.csv'), TIMEPOINTS)
print(f'Phosphosite matrix : {Y.shape}  (N_sites × T)')
print(f'Protein abundance  : {A_data.shape}  (K × T)')
print(f'Proteins: {proteins}')
print(f'Sites   : {sites}')
print(f'Time    : {t_phos}')

In [ ]:
gene_ids, t_rna, rna_matrix = load_rna_data(
    str(SAMPLE_DIR / 'mrna.csv'), timepoints=TIMEPOINTS)
print(f'RNA matrix  : {rna_matrix.shape}  (n_genes × T)')
print(f'RNA genes   : {gene_ids}')

K_site_kin, kinases = load_kinase_site_matrix(
    str(SAMPLE_DIR / 'kinase_sites.tsv'), sites)
print(f'K_site_kin  : {K_site_kin.shape}  (N_sites × M_kinases)')
print(f'Kinases     : {kinases}')

## 3d. Scaling, Weights, and Derived Structures

In [ ]:
P_scaled, _, _ = apply_scaling(Y)
A_scaled, _, _ = apply_scaling(A_data)
rna_scaled = row_normalize(rna_matrix)

print(f'P_scaled range   : [{P_scaled.min():.3f}, {P_scaled.max():.3f}]')
print(f'A_scaled range   : [{A_scaled.min():.3f}, {A_scaled.max():.3f}]')

tf_net = load_tf_network(str(SAMPLE_DIR / 'tf_mrna.csv'), gene_ids=gene_ids)
tf_prot_weights = build_tf_prot_weights(tf_net, gene_ids, proteins)
print(f'tf_prot_weights  : {tf_prot_weights.shape}  (K × n_genes)')

## 3e. ModelDims, Bounds, Parameter Labels

In [ ]:
from phoscrosstalk.config import ModelDims
from phoscrosstalk.optimization import create_bounds, build_parameter_labels

K = len(proteins);  M = len(kinases);  N = len(sites)
dims = ModelDims(K=K, M=M, N=N)

xl, xu, n_params = create_bounds(K, M, N)
theta_names = build_parameter_labels(K, M, N)
print(f'ModelDims: K={K}, M={M}, N={N}')
print(f'n_params = {n_params}')
print(f'First 8 names: {theta_names[:8]}')

## 3f. Build Rate Closures

In [ ]:
from phoscrosstalk.derived_rates import make_k_act_fn, make_s_prod_fn

kin_to_prot_idx = np.array([proteins.index(k) for k in kinases], dtype=int)
R_kin = row_normalize(K_site_kin.T)

k_act_fn  = make_k_act_fn(t_rna=t_rna, rna_data=rna_scaled,
                           tf_prot_weights=tf_prot_weights, K=K)
s_prod_fn = make_s_prod_fn(t_protein=t_phos, Y_data=P_scaled,
                            R_kin_site=R_kin,
                            kin_to_prot_idx=kin_to_prot_idx, K=K, M=M)
print('k_act_fn  : created')
print('s_prod_fn : created')

## 3g. Single Forward Simulation to Verify Setup

In [ ]:
from phoscrosstalk.simulation import simulate

Cg = np.zeros((N, N));  Cl = np.zeros((N, N))
L_alpha = np.zeros((M, M))
receptor_mask_prot = np.zeros(K);  receptor_mask_kin = np.zeros(M)

theta_init = rng.uniform(xl, xu)

P_sim, A_sim, S_sim, Kdyn_sim = simulate(
    t_arr=t_phos, P_data0=P_scaled, A_data0=A_scaled,
    theta=theta_init, Cg=Cg, Cl=Cl,
    site_prot_idx=site_prot_idx, K_site_kin=K_site_kin, R=R_kin,
    L_alpha=L_alpha, kin_to_prot_idx=kin_to_prot_idx,
    receptor_mask_prot=receptor_mask_prot, receptor_mask_kin=receptor_mask_kin,
    mechanism='dist', full_output=True,
    k_act_fn=k_act_fn, s_prod_fn=s_prod_fn,
)
print(f'✓ Simulation succeeded')
print(f'  P_sim : {P_sim.shape}  (should be ({N}, {len(t_phos)}))')
print(f'  A_sim : {A_sim.shape}  (should be ({K}, {len(t_phos)}))')
print(f'  NaN in P_sim: {np.isnan(P_sim).sum()}')

In [ ]:
fig, axes = plt.subplots(2, min(4, max(K, N)), figsize=(14, 7), squeeze=False)

for i in range(min(4, N)):
    axes[0][i].plot(t_phos, P_scaled[i], 'ko-', ms=5, label='data')
    axes[0][i].plot(t_phos, P_sim[i],    'b--', lw=2,  label='sim (random θ)')
    axes[0][i].set_title(sites[i], fontsize=9)
    axes[0][i].set_xlabel('Time');  axes[0][i].set_ylabel('p-site')
    if i == 0: axes[0][i].legend(fontsize=8)

for i in range(K):
    axes[1][i].plot(t_phos, A_scaled[i], 'ko-', ms=5, label='data')
    axes[1][i].plot(t_phos, A_sim[i],    'r--', lw=2,  label='sim (random θ)')
    axes[1][i].set_title(proteins[i], fontsize=9)
    axes[1][i].set_xlabel('Time');  axes[1][i].set_ylabel('Protein')
    if i == 0: axes[1][i].legend(fontsize=8)

plt.suptitle('Forward simulation at random θ (before fitting)', fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '15_forward_simulation_check.png', dpi=120)
plt.show()
print('Saved: 15_forward_simulation_check.png')
print('✓ Pipeline verified — ready for optimisation')

## 4. From Here: Running the Optimisation

**Option A \u2014 CLI (recommended):**
```bash
phoscrosstalk --config config.toml
```

**Option B \u2014 Python API:**
```python
import argparse
from phoscrosstalk.multistarts import run_multi_start_optimization

# Build a minimal args namespace matching the CLI contract
args = argparse.Namespace(
    n_starts=20,
    max_steps=500,
    backend='optimistix',
    ls_solver='lm',
    jac_mode='fwd',
    parallel_starts=1,
    threads_per_start=1,
    # ... any other fields your build of phoscrosstalk requires
)

merged_res, best_idx, total_losses = run_multi_start_optimization(
    problem,    # NetworkProblem
    args,       # argparse.Namespace with n_starts, max_steps, backend, ...
    P_scaled,   # (N, T) scaled phosphosite data
)
theta_best = merged_res.X[best_idx]
```


## 5. Expected Output Directory Structure

```
results/
├── theta_best.npy
├── fit_timeseries.tsv
├── multistart_results.tsv
├── plots/
│   ├── phospho_fit.png
│   └── protein_fit.png
├── uncertainty/          # [posterior] enabled=true
│   ├── bootstrap_samples.npz
│   ├── bootstrap_summary.tsv
│   └── profile_likelihood_plots/
├── steadystate/          # run_steadystate=true
│   └── steady_state_results.tsv
├── sensitivity/          # run_sensitivity=true
│   └── sensitivity_heatmap.png
└── knockouts/            # run_knockouts=true
    └── knockout_*.png
```

## 6. Common Pitfalls & Troubleshooting

| Problem | Cause | Fix |
|---------|-------|-----|
| `phosphosite_time_points` mismatch | # columns ≠ time list length | Count x-columns, update config |
| `protein_time_points` mismatch | Must equal phosphosite_time_points | Use same list |
| Kinase names don't match | Casing or symbol difference | Standardise GeneID casing |
| `K_site_kin` all zeros | No kinase/gene overlap | Check name matching |
| NaN in loss | NaN in input data | `df.fillna()` or drop rows |
| Inf in loss | Zero division in ODE | Tighten bounds in `create_bounds()` |

In [ ]:
print('=' * 55)
print('PhosCrosstalk pipeline — setup summary')
print('=' * 55)
print(f'  Proteins (K={K}): {proteins}')
print(f'  Kinases  (M={M}): {kinases}')
print(f'  P-sites  (N={N}): {sites}')
print(f'  Time pts (T={len(t_phos)}): {list(t_phos)}')
print(f'  n_params = {n_params}')
print(f'  K_site_kin non-zero: {int((K_site_kin != 0).sum())}/{K_site_kin.size}')
print(f'  NaN in Y      : {np.isnan(Y).sum()}')
print(f'  NaN in A_data : {np.isnan(A_data).sum()}')
status = 'OK' if not np.isnan(P_sim).any() else 'NaN detected!'
print(f'  Simulation    : {status}')
print('=' * 55)